In [ ]:
print("\n--- NEW METRICS WITH THE CORRECTED CALL MISJUDGED BY UMPIRE ---")

# --- Metadata including Ground Truth TV Strike Zones ---
video_metadata = {
    '0001.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Ohtani, Shoei', 'count': '0 - 0', 'true_pitch_result': 'STRIKE', 'tv_zone': (660, 240, 730, 325)},
    '0002.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Ohtani, Shoei', 'count': '0 - 1', 'true_pitch_result': 'STRIKE', 'tv_zone': (660, 240, 730, 325)},
    '0003.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Betts, Mookie', 'count': '0 - 0', 'true_pitch_result': 'STRIKE', 'tv_zone': (660, 240, 730, 325)},
    '0004.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Freeman, Freddie', 'count': '0 - 0', 'true_pitch_result': 'BALL', 'tv_zone': (660, 235, 730, 320)},
    '0005.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Freeman, Freddie', 'count': '1 - 1', 'true_pitch_result': 'BALL', 'tv_zone': (660, 228, 727, 318)},
    '0006.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Freeman, Freddie', 'count': '2 - 1', 'true_pitch_result': 'BALL', 'tv_zone': (665, 228, 725, 315)},
    '0007.mp4': {'pitcher': 'Cole, Gerrit', 'hitter': 'Freeman, Freddie', 'count': '3 - 1', 'true_pitch_result': 'BALL', 'tv_zone': (660, 228, 725, 325)},
    '0008.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Tatis Jr., Fernando', 'count': '0 - 1', 'true_pitch_result': 'BALL', 'tv_zone': (710,250, 770, 333)},
    '0009.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Tatis Jr., Fernando', 'count': '1 - 1', 'true_pitch_result': 'BALL', 'tv_zone': (710,250, 770, 333)},
    '0010.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Tatis Jr., Fernando', 'count': '2 - 1', 'true_pitch_result': 'BALL', 'tv_zone': (710,250, 770, 333)},
    '0011.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Arraez, Luis', 'count': '2 - 1', 'true_pitch_result': 'STRIKE', 'tv_zone': (710,285, 770, 355)},
    '0012.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Sheets, Gavin', 'count': '0 - 1', 'true_pitch_result': 'STRIKE', 'tv_zone': (710,270, 770, 342)},
    '0013.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Bogaerts, Xander', 'count': '0 - 0', 'true_pitch_result': 'STRIKE', 'tv_zone': (710,260, 765, 340)},
    '0014.mp4': {'pitcher': 'Ohtani, Shoei', 'hitter': 'Wood, James', 'count': '0 - 0', 'true_pitch_result': 'STRIKE', 'tv_zone': (710,250, 770, 333)},
}
final_results = []
iou_scores = []

def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def line_intersects_rect(p1, p2, rect):
    """Checks if the line segment from p1 to p2 intersects with rect."""
    x1, y1 = p1
    x2, y2 = p2
    rx, ry, rw, rh = rect[0], rect[1], rect[2]-rect[0], rect[3]-rect[1]
    
    # Check if either endpoint is inside the rect
    if (rx <= x1 <= rx + rw and ry <= y1 <= ry + rh) or \
       (rx <= x2 <= rx + rw and ry <= y2 <= ry + rh):
        return True

    # Check for intersection with each of the 4 rectangle sides
    for i in range(4):
        x3, y3 = (rx, ry) if i == 0 else (rx + rw, ry) if i == 1 else (rx + rw, ry + rh) if i == 2 else (rx, ry + rh)
        x4, y4 = (rx + rw, ry) if i == 0 else (rx + rw, ry + rh) if i == 1 else (rx, ry + rh) if i == 2 else (rx, ry)
        
        den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if den == 0:
            continue
        
        t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / den
        u = -((x1 - x2) * (y1 - y3) - (y1 - y2) * (x1 - x3)) / den
        
        if 0 < t < 1 and 0 < u < 1:
            return True
            
    return False

if video_paths and plate_detector_model and hitter_detector_model:
    generic_pose_model = YOLO('yolov8n-pose.pt')

    for video_path in video_paths:
        if video_path not in video_metadata:
            continue

        print(f"\n\n--- Processing video: {video_path} ---")

        cap = cv2.VideoCapture(video_path)
        width, height, fps = (int(cap.get(c)) for c in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

        input_video_name = Path(video_path).stem
        output_video_path = PROJECT_ROOT / f'{input_video_name}_output.mp4'
        out = cv2.VideoWriter(str(output_video_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

        frozen_strike_zone = None
        frozen_tv_zone = video_metadata[video_path].get('tv_zone')

        trajectories = {}
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_count += 1
            print(f"  Processing frame {frame_count}...")

            track_results = plate_detector_model.track(frame, persist=True, verbose=False)

            if frozen_strike_zone is None and track_results[0].boxes.id is not None:
                for box, track_id, cls_id in zip(track_results[0].boxes, track_results[0].boxes.id.int().cpu().tolist(), track_results[0].boxes.cls.int().cpu().tolist()):
                    if plate_detector_model.names[cls_id] == 'baseball' and box.conf[0] > 0.5:
                        print(f"  Ball first detected at frame {frame_count}. Freezing strike zone...")

                        hitter_results = hitter_detector_model(frame, verbose=False)
                        home_plate_box = next((b.xyxy[0].cpu().numpy() for b in track_results[0].boxes if plate_detector_model.names[int(b.cls[0])] == 'homeplate'), None)
                        hitter_box = next((b.xyxy[0].cpu().numpy() for b in hitter_results[0].boxes if hitter_detector_model.names[int(b.cls[0])] == 'hitter'), None)

                        if home_plate_box is not None and hitter_box is not None:
                            x1_h, y1_h, x2_h, y2_h = [int(c) for c in hitter_box]
                            hitter_crop = frame[y1_h:y2_h, x1_h:x2_h]
                            pose_results = generic_pose_model(hitter_crop, verbose=False)
                            if pose_results[0].keypoints:
                                hitter_keypoints = pose_results[0].keypoints.xy[0].cpu().numpy()
                                hitter_keypoints[:, 0] += x1_h; hitter_keypoints[:, 1] += y1_h
                                sz_left = int(home_plate_box[0]); sz_right = int(home_plate_box[2])
                                plate_top_y = int(home_plate_box[1])
                                shoulder_y = (hitter_keypoints[5][1] + hitter_keypoints[6][1]) / 2
                                hip_y = (hitter_keypoints[11][1] + hitter_keypoints[12][1]) / 2
                                sz_top = int((shoulder_y + hip_y) / 2)
                                sz_bottom = int(max(hitter_keypoints[13][1], hitter_keypoints[14][1]))
                                sz_bottom = min(sz_bottom, plate_top_y)
                                if sz_bottom > sz_top > 0:
                                    frozen_strike_zone = (sz_left, sz_top, sz_right, sz_bottom)
                                    print(f"  Strike Zone FROZEN: {frozen_strike_zone}")
                                    if frozen_tv_zone and frozen_tv_zone != (0,0,0,0):
                                        iou = calculate_iou(frozen_strike_zone, frozen_tv_zone)
                                        iou_scores.append(iou)
                                        print(f"  IoU Calculated: {iou:.2%}")
                        break

            if track_results[0].boxes.id is not None:
                for box, track_id, cls_id in zip(track_results[0].boxes, track_results[0].boxes.id.int().cpu().tolist(), track_results[0].boxes.cls.int().cpu().tolist()):
                    if plate_detector_model.names[cls_id] == 'baseball':
                        center_x = int((box.xyxy[0][0] + box.xyxy[0][2]) / 2)
                        center_y = int((box.xyxy[0][1] + box.xyxy[0][3]) / 2)
                        if track_id not in trajectories:
                            trajectories[track_id] = []
                        trajectories[track_id].append((center_x, center_y))

            if frozen_strike_zone:
                cv2.rectangle(frame, (frozen_strike_zone[0], frozen_strike_zone[1]), (frozen_strike_zone[2], frozen_strike_zone[3]), (0, 255, 255), 2)

            if frozen_tv_zone and frozen_tv_zone != (0,0,0,0):
                tv_l, tv_t, tv_r, tv_b = frozen_tv_zone
                for i in range(tv_t, tv_b, 15): cv2.line(frame, (tv_l, i), (tv_l, i+10), (255,255,255), 2)
                for i in range(tv_t, tv_b, 15): cv2.line(frame, (tv_r, i), (tv_r, i+10), (255,255,255), 2)
                for i in range(tv_l, tv_r, 15): cv2.line(frame, (i, tv_t), (i+10, tv_t), (255,255,255), 2)
                for i in range(tv_l, tv_r, 15): cv2.line(frame, (i, tv_b), (i+10, tv_b), (255,255,255), 2)

            if frozen_strike_zone and frozen_tv_zone and frozen_tv_zone != (0,0,0,0):
                inter_x1 = max(frozen_strike_zone[0], frozen_tv_zone[0]); inter_y1 = max(frozen_strike_zone[1], frozen_tv_zone[1])
                inter_x2 = min(frozen_strike_zone[2], frozen_tv_zone[2]); inter_y2 = min(frozen_strike_zone[3], frozen_tv_zone[3])
                if inter_x2 > inter_x1 and inter_y2 > inter_y1:
                    overlay = frame.copy()
                    cv2.rectangle(overlay, (inter_x1, inter_y1), (inter_x2, inter_y2), (0, 255, 0), -1)
                    cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)

            if trajectories:
                longest_trajectory_id = max(trajectories, key=lambda k: len(trajectories[k]))
                if len(trajectories[longest_trajectory_id]) > 1:
                    path = trajectories[longest_trajectory_id]
                    cv2.polylines(frame, [np.array(path, np.int32)], isClosed=False, color=(255, 0, 0), thickness=2)
            out.write(frame)

        # --- Make the final call ---
        final_call_text = "INCONCLUSIVE"
        is_strike = False
        if frozen_strike_zone and trajectories:
            longest_trajectory_id = max(trajectories, key=lambda k: len(trajectories[k]))
            ball_trajectory = trajectories[longest_trajectory_id]

            # Checking for line segment intersection
            if len(ball_trajectory) > 1:
                for i in range(len(ball_trajectory) - 1):
                    p1 = ball_trajectory[i]
                    p2 = ball_trajectory[i+1]
                    if line_intersects_rect(p1, p2, frozen_strike_zone):
                        is_strike = True
                        break
            final_call_text = "STRIKE" if is_strike else "BALL"

        current_video_info = video_metadata[video_path]
        current_video_info['model_call'] = final_call_text
        current_video_info['filename'] = Path(video_path).name
        final_results.append(current_video_info)

        cap.release()
        out.release()
        print(f"\nVideo processing complete for '{video_path}'. Output saved to: {output_video_path}")

    # --- Print the final summary table and metrics ---
    final_results.sort(key=lambda x: x['filename'])
    print("\n\n--- FINAL RESULTS SUMMARY ---")
    print("\nPITCHER: GERRIT COLE")
    for res in final_results:
        filename = res['filename']
        if int(Path(filename).stem) <= 7:
             print(f"{filename} Pitcher: {res['pitcher']:<20} | Hitter: {res['hitter']:<20} | Count: {res['count']:<5} | True pitch result: {res['true_pitch_result']:<7} | Model Call: {res['model_call']}")

    print("\nPITCHER: SHOEI OHTANI")
    for res in final_results:
        filename = res['filename']
        if int(Path(filename).stem) > 7:
             print(f"{filename} Pitcher: {res['pitcher']:<20} | Hitter: {res['hitter']:<20} | Count: {res['count']:<5} | True pitch result: {res['true_pitch_result']:<7} | Model Call: {res['model_call']}")

    y_true = [res['true_pitch_result'] for res in final_results]
    y_pred = [res['model_call'].split(' ')[0] for res in final_results]

    if y_true and y_pred:
        tp, fp, tn, fn = 0, 0, 0, 0
        for official, model in zip(y_true, y_pred):
            if model == "STRIKE" and official == "STRIKE": tp += 1
            elif model == "STRIKE" and official == "BALL": fp += 1
            elif model == "BALL" and official == "BALL": tn += 1
            elif model == "BALL" and official == "STRIKE": fn += 1

        total_calls = len(final_results)
        accuracy = (tp + tn) / total_calls if total_calls > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        print("\n\n--- PERFORMANCE METRICS ---")
        print(f"Total Pitches Analyzed: {total_calls}")
        print(f"Accuracy:  {accuracy:.2%}")
        print(f"Precision: {precision:.2%}")
        print(f"Recall:    {recall:.2%}")
        print(f"F1-Score:  {f1_score:.2%}")

        if iou_scores:
            mean_iou = np.mean(iou_scores)
            print(f"Mean Intersection over Union (IoU): {mean_iou:.2%}")

        print("--------------------------")

        # --- Plot Confusion Matrix ---
        cm = confusion_matrix(y_true, y_pred, labels=["STRIKE", "BALL"])
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=["STRIKE", "BALL"], yticklabels=["STRIKE", "BALL"])
        plt.xlabel('Model Call')
        plt.ylabel('Official Call')
        plt.title('Confusion Matrix')
        plt.show()